# Microservices-based Financial Data API with CI/CD & Unit Testing

Step 1: Project Setup & Class Architecture
In this step, we define our core financial transaction model using Pydantic for schema validation and create our transactional database engine.

In [1]:
# Install required dependencies
!pip install -q pydantic fastapi uvicorn

import sqlite3
from typing import List, Optional
from pydantic import BaseModel, Field

# 1. Define Financial Transaction Data Schema
class Transaction(BaseModel):
    transaction_id: str
    account_id: str
    amount: float = Field(..., gt=0, description="Transaction amount must be positive")
    currency: str = Field(default="USD", max_length=3)
    category: str

# 2. Database Component Class
class FinancialDatabase:
    def __init__(self, db_name=":memory:"):
        self.conn = sqlite3.connect(db_name)
        self.cursor = self.conn.cursor()
        self._create_table()

    def _create_table(self):
        self.cursor.execute("""
            CREATE TABLE IF NOT EXISTS transactions (
                transaction_id TEXT PRIMARY KEY,
                account_id TEXT,
                amount REAL,
                currency TEXT,
                category TEXT
            )
        """)
        self.conn.commit()

    def insert_transaction(self, tx: Transaction) -> bool:
        try:
            self.cursor.execute("""
                INSERT INTO transactions (transaction_id, account_id, amount, currency, category)
                VALUES (?, ?, ?, ?, ?)
            """, (tx.transaction_id, tx.account_id, tx.amount, tx.currency, tx.category))
            self.conn.commit()
            return True
        except sqlite3.IntegrityError:
            return False

    def get_account_summary(self, account_id: str) -> dict:
        self.cursor.execute("""
            SELECT COUNT(*), SUM(amount) FROM transactions WHERE account_id = ?
        """, (account_id,))
        count, total = self.cursor.fetchone()
        return {
            "account_id": account_id,
            "total_transactions": count,
            "total_volume": total if total else 0.0
        }

print("Step 1 Complete: Database engine and schemas configured successfully.")

Step 1 Complete: Database engine and schemas configured successfully.


Step 2: Service Layer & Exception Handling
Here, we build the service layer that processes financial rules (e.g., currency checking, transaction volume limits) and logs operational errors.

In [2]:
class FinancialService:
    def __init__(self, db: FinancialDatabase):
        self.db = db

    def process_transaction(self, tx_data: dict) -> dict:
        # Validate input schema
        tx = Transaction(**tx_data)

        # Business Rule: Rejection of amounts over $100,000 without manual review
        if tx.amount > 100000:
            raise ValueError("Transaction exceeds maximum automated limit ($100,000)")

        # Save to database
        success = self.db.insert_transaction(tx)
        if not success:
            raise ValueError(f"Duplicate transaction ID: {tx.transaction_id}")

        return {"status": "SUCCESS", "transaction_id": tx.transaction_id}

print("Step 2 Complete: Business logic and validation layer active.")

Step 2 Complete: Business logic and validation layer active.


Step 3: Automated Unit Testing Suite
To satisfy IBM's requirement for software quality and unit testing, we write tests using unittest to verify both successful transactions and edge cases.

In [3]:
import unittest

class TestFinancialService(unittest.TestCase):
    def setUp(self):
        self.db = FinancialDatabase(":memory:")
        self.service = FinancialService(self.db)

    def test_valid_transaction(self):
        sample_tx = {
            "transaction_id": "TX1001",
            "account_id": "ACC_99",
            "amount": 2500.50,
            "currency": "USD",
            "category": "Operations"
        }
        res = self.service.process_transaction(sample_tx)
        self.assertEqual(res["status"], "SUCCESS")

    def test_exceed_limit_rejection(self):
        large_tx = {
            "transaction_id": "TX1002",
            "account_id": "ACC_99",
            "amount": 500000.00,
            "currency": "USD",
            "category": "Capital"
        }
        with self.assertRaises(ValueError):
            self.service.process_transaction(large_tx)

    def test_duplicate_transaction_id(self):
        sample_tx = {
            "transaction_id": "TX1003",
            "account_id": "ACC_01",
            "amount": 100.00,
            "currency": "USD",
            "category": "Retail"
        }
        self.service.process_transaction(sample_tx)
        with self.assertRaises(ValueError):
            self.service.process_transaction(sample_tx)

# Run unit tests directly inside Colab
suite = unittest.TestLoader().loadTestsFromTestCase(TestFinancialService)
runner = unittest.TextTestRunner(verbosity=2)
runner.run(suite)

test_duplicate_transaction_id (__main__.TestFinancialService.test_duplicate_transaction_id) ... ok
test_exceed_limit_rejection (__main__.TestFinancialService.test_exceed_limit_rejection) ... ok
test_valid_transaction (__main__.TestFinancialService.test_valid_transaction) ... ok

----------------------------------------------------------------------
Ran 3 tests in 0.010s

OK


<unittest.runner.TextTestResult run=3 errors=0 failures=0>

Step 4: GitHub CI/CD Pipeline Automation Script
To demonstrate understanding of SDLC and CI/CD pipelines, this script generates a .github/workflows/ci.yml pipeline configuration file directly from Colab.

In [4]:
import os

# Create directory structure for GitHub Actions
os.makedirs(".github/workflows", exist_ok=True)

github_actions_workflow = """
name: Financial API CI/CD Pipeline

on:
  push:
    branches: [ "main" ]
  pull_request:
    branches: [ "main" ]

jobs:
  build-and-test:
    runs-on: ubuntu-latest

    steps:
    - uses: actions/checkout@v3

    - name: Set up Python 3.10
      uses: actions/setup-python@v4
      with:
        python-version: "3.10"

    - name: Install Dependencies
      run: |
        python -m pip install --upgrade pip
        pip install pydantic fastapi pytest

    - name: Run Automated Unit Tests
      run: |
        pytest
"""

with open(".github/workflows/ci.yml", "w") as f:
    f.write(github_actions_workflow.strip())

print("Step 4 Complete: GitHub CI/CD configuration written to .github/workflows/ci.yml")

Step 4 Complete: GitHub CI/CD configuration written to .github/workflows/ci.yml
